In [ ]:
!pip install git+https://github.com/huggingface/transformers
!pip install --upgrade unsloth unsloth_zoo
!pip install --upgrade --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-rnr5hpyl
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-rnr5hpyl
  Resolved https://github.com/huggingface/transformers to commit c7cf04b1e3b1d497dbb1473c2e65e75ee69e12dc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.0 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.16.0.dev0-py3-none-any.whl size=13156571 sha256=b8f2471e6d835840c8ba5672bab4fbfd5a5c9f2a2145c86617f10ac3411d03da
  Stored in directory: /tmp/pip-ephem-wheel-cache-x7ths9m7/wheels/2b/de/48/1c5b158806820c3979e1dc7a341b68dac1231f00b1a2c9442f
Successfully built transformers
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully 

In [17]:
from pathlib import Path
import os
from dotenv import load_dotenv, dotenv_values
import json
from PIL import Image
from datasets import Dataset

In [ ]:
!unzip annotations.zip
!unzip company_filing_pages.zip

Archive:  annotations.zip
   creating: annotations/
  inflating: annotations/01436782.json  
  inflating: annotations/01612140.json  
  inflating: annotations/01903832.json  
  inflating: annotations/02206597.json  
  inflating: annotations/02278974.json  
  inflating: annotations/02291274.json  
  inflating: annotations/02452303.json  
  inflating: annotations/02526128.json  
  inflating: annotations/02624500.json  
  inflating: annotations/02859692.json  
  inflating: annotations/03004352.json  
  inflating: annotations/03743696.json  
  inflating: annotations/03825186.json  
  inflating: annotations/04091297.json  
  inflating: annotations/04465268.json  
  inflating: annotations/04774581.json  
  inflating: annotations/05155587.json  
  inflating: annotations/05241943.json  
  inflating: annotations/05536016.json  
  inflating: annotations/05993442.json  
  inflating: annotations/06481116.json  
  inflating: annotations/06617562.json  
  inflating: annotations/06628106.json  
  inf

In [ ]:
INSTRUCTION = """Look at this image and determine if it contains an actual table — that is, structured content with visible rows
                  and columns, gridlines or clear column alignment, and multiple data fields per row. There may be zero, one, or
                  multiple seperate tables in this page, so don't stop after finding just one. Tables are usually seperated by
                  headers, whitespace, or a change in column structure. Extract them as seperate tables.

                  OUTPUT - the output should always be the "table_found" field followed by a  list of tables, even if there is just one.

                  Do NOT treat the following as tables:
                  - Body text or prose paragraphs, even if organized under headings
                  - Single-column lists of headings followed by descriptive text
                  - Definition-style content (a term followed by an explanatory paragraph)

                  If NO real table is present, respond with exactly:
                  {"table_found": false}

                  If a real table IS present, extract it with this format:
                  {
                    "table_found": true,
                    "tables" : [
                    {
                    "table_name": "<inferred from caption/context>",
                    "headers" : [header1, header2, ...],
                    "rows": [
                      ["row1val1", "row1val2", ...]
                        ]
                      }
                    ]
                  }


                  Rules:
                  - Output valid JSON only, no explanation before or after
                  - Preserve merged cells by repeating the value across merged rows/columns
                  - If a cell is empty or unreadable, use null
                  - Keep numbers as strings if they include currency symbols, %, or commas
                  - Always extract exactly what appears in the top row, cell by cell, even if it's just a date or number (i.e. Q3 2025 or 2026)
                  - In addition to the last rule, if a row's first cell is a date, "e.g. "At 31 December 2022", and has a number (e.g. 25,000 2300) in the same row, it is ALWAYS a data row and is to NEVER be a header — this is true even if it's position is the same as the header's
                  - A header row's cells are category or period labels only — they never carry a specific numeric value
                  - If there is no distinct header row at all, set column_headers to null
                  """

In [20]:
load_dotenv(Path.cwd() / ".env")

True

In [ ]:
ANNOTATIONS = Path.cwd().parent / "DataPreprocess" / "annotations"
COMPANY_PAGES = Path.cwd().parent / "DataPreprocess" / "company_filing_pages"
annotated_com = [Path(i).stem for i in os.listdir(ANNOTATIONS)]

In [ ]:
split_index = int(len(annotated_com) * 0.8)

train_com = annotated_com[:split_index]
test_com = annotated_com[split_index:]

In [ ]:
def create_company_index(companies: list):
  out = []
  for i in companies:
    with open(ANNOTATIONS / (i + ".json"), "r") as f:
      pages = json.load(f)

    for page, payload in pages.items():
      image_path = COMPANY_PAGES / i / page

      out.append(
          {
              "image": str(image_path),
              "output": json.dumps(payload["payload"])
          }
      )

  return out

def load_example(ex):
    ex["image"] = Image.open(ex["image"]).convert("RGB")
    return ex

def convert_to_conversation(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": example["image"],
                    },
                    {
                        "type": "text",
                        "text": INSTRUCTION
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": example["output"]
                    }
                ]
            }
        ]
    }


train_set = Dataset.from_list(create_company_index(train_com))
test_set = Dataset.from_list(create_company_index(test_com))

train_set = train_set.map(load_example).map(convert_to_conversation)
test_set = test_set.map(load_example).map(convert_to_conversation)


print(f"{len(train_set)} examples loaded into Train Set")
print(f"{len(test_set)} examples loaded into Test Set")

Map:   0%|          | 0/219 [00:00<?, ? examples/s]

Map:   0%|          | 0/219 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

219 examples loaded into Train Set
45 examples loaded into Test Set


In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_set,
    eval_dataset=test_set,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=1,
        optim="adamw_8bit",
        output_dir="qwen3_vl_finetuned",
        remove_unused_columns=False,
        dataset_text_field="",
        max_seq_length=2048,
        eval_strategy ="steps",
        load_best_model_at_end=True
    ),
)

trainer.train()

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 219 | Num Epochs = 3 | Total steps = 84
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 21,823,488 of 8,788,947,184 (0.25% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
1,3.917313,3.766742
2,3.772986,3.721930
3,3.514332,3.478716
4,3.345855,2.892308
5,2.387476,2.309072
6,1.985844,2.036568
7,1.859468,1.920469
8,2.000237,1.824057
9,1.794389,1.734473
10,1.638440,1.642114


Unsloth: Restored added_tokens_decoder metadata in qwen3_vl_finetuned/checkpoint-84/tokenizer_config.json.


TrainOutput(global_step=84, training_loss=0.45610864123946593, metrics={'train_runtime': 3600.451, 'train_samples_per_second': 0.182, 'train_steps_per_second': 0.023, 'total_flos': 3.459254046717888e+16, 'train_loss': 0.45610864123946593, 'epoch': 3.0})

In [ ]:
print(trainer.state.best_model_checkpoint) 
print(trainer.state.best_metric)

None
0.0035837097093462944


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
MODEL_SAVE_PATH = os.getenv("MODEL_SAVE_PATH")
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/qwen3_vl_lora/tokenizer_config.json.


['/content/drive/MyDrive/qwen3_vl_lora/processor_config.json']

In [ ]:
test_com

['04774581',
 '02526128',
 '13700075',
 '15229106',
 '10192734',
 '02291274',
 '06617562',
 '14361496',
 '13879337',
 '03004352',
 'SC454219',
 '02859692']